# Pseudotime analysis

In [ ]:
import scanpy as sc
import scvi
import palantir as pt
import cellrank as cr

import pickle

from scipy import sparse
import pandas as pd
import numpy as np

import os

# Setup 

In [ ]:
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.1.0-FD20220623HSCT/lib/R'

In [ ]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri

In [ ]:
%load_ext rpy2.ipython

## Settings

In [ ]:
# Working dir
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [ ]:
# Warnings Python 
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning) 

# Warnings scanpy off
sc.settings.verbosity = 0

In [ ]:
# CPU 
scvi.settings.num_threads=48

## Plotting 

In [ ]:
# Scanpy figures
sc.set_figure_params(figsize=(5, 5), dpi_save=1200, fontsize=12, frameon=False, color_map='viridis', facecolor='white')

# Import AnnData 

In [ ]:
adata = sc.read_h5ad('data/object/pp_subset.h5ad')

# Helper functions

## SCVI integration 

In [ ]:
def scvi_integrate(adata, model_dir=None, file=None): 
    
    adata.raw = adata
    adata = adata[:, (adata.X>=3).sum(axis=0)>=3].copy()
    
    scvi.model.SCVI.setup_anndata(
        
        adata,
        batch_key='sample_name'
        
    )
    
    model = scvi.model.SCVI(
        
        adata, 
        n_latent=10, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'
        
    )
    
    model.train()
    
    # Write output 
    if model_dir is not None: 
        model.save(model_dir, overwrite=True)
    
    adata.obsm['X_scvi'] = model.get_latent_representation()
    
    # Write output 
    if file is not None: 
        adata.write_h5ad(file)
    
    return adata

## PAGA dimensional reduction and pseudotime computation 

In [ ]:
def paga_dim(adata, n_neighbors=15, use_rep='X_scvi', metric='euclidean', resolution=2, groups='leiden', layout='fr'):
    
    # SCVI based neighbors and Leiden cluster
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep=use_rep, metric=metric)
    sc.tl.leiden(adata, resolution=resolution)
        
    # PAGA 
    sc.tl.paga(adata, groups=groups, model='v1.2')
    sc.pl.paga(adata, layout=layout, threshold=0, plot=False)
    
    # PAGA UMAP
    sc.tl.umap(adata, init_pos='paga', min_dist=0.5)
    
    return adata

In [ ]:
def palantir_workflow(adata, celltype, celltype_column, n_components=15, knn=30): 
    
    # Save original count data 
    X = adata.X
    
    # Normalization 
    sc.pp.normalize_per_cell(adata)
    pt.preprocess.log_transform(adata)
    adata.layers['data'] = adata.X
    
    sc.pp.pca(adata)
    
    # Compute diffusion components
    dm_res = pt.utils.run_diffusion_maps(adata, n_components=n_components, knn=knn, pca_key='X_pca')
    
    # Data imputation with MAGIC
    adata.X=sparse.csr_matrix(adata.X)
    adata.layers['data_magic'] = sparse.csr_matrix(pt.utils.run_magic_imputation(adata, dm_res))
    
    # Low dimensional embedding of the data based on the eigen gap
    ms_data = pt.utils.determine_multiscale_space(adata)
    adata.obsm['X_diffmap'] = ms_data.to_numpy()
    
    # Compute start cell
    early_cell_id = pt.utils.early_cell(adata, celltype, celltype_column=celltype_column)
    
    # Running Palantir 
    pr_res = pt.core.run_palantir(adata, early_cell=early_cell_id, knn=knn)
    
    # Set count data to X
    adata.X = X
    
    return adata

In [ ]:
def module_score(adata): 
    
    X = adata.X.copy()

    sc.pp.normalize_total(adata, target_sum=1e6)
    sc.pp.log1p(adata)

    sc.tl.score_genes(adata, gene_list=['CCR6'], score_name='ms_ccr6', use_raw=False, n_bins=25, ctrl_size=100)
    sc.tl.score_genes(adata, gene_list=['IL7R'], score_name='ms_il7r', use_raw=False, n_bins=25, ctrl_size=100)
    sc.tl.score_genes(adata, gene_list=['CCR6', 'IL7R'], score_name='ms_ccr6_il7r', use_raw=False, n_bins=25, ctrl_size=100)
    
    adata.X = X
    
    return adata

In [ ]:
def pl_summary(adata, layer='data'): 
    
    # Set layer for plotting 
    adata.X = adata.layers[layer]
    
    # Plot 
    sc.pl.paga(adata, layout='fr', threshold=0.25, fontsize=12, node_size_scale=1.0, node_size_power=1.0, max_edge_width=0.5, color=['cell_type_leiden_subset', 'hto_demux_class', 'time_point', 'genotype_class', 'has_ir', 'palantir_pseudotime'])
    sc.pl.paga(adata, layout='fr', threshold=0.25, fontsize=12, node_size_scale=1.0, node_size_power=1.0, max_edge_width=0.5, color=['CCR6', 'MAF', 'IL7R', 'IL17A', 'RUNX3', 'ZNF683'], use_raw=True)
    sc.pl.umap(adata, color=['cell_type_leiden_subset', 'hto_demux_class', 'time_point', 'genotype_class', 'has_ir', 'palantir_pseudotime'], wspace=0.2, ncols=6)
    sc.pl.embedding(adata, basis='X_diffmap', color=['cell_type_leiden_subset', 'hto_demux_class', 'time_point', 'genotype_class', 'has_ir', 'palantir_pseudotime'], wspace=0.2, ncols=6)

## CellRank 

In [ ]:
def kernel_workflow(adata, pk_ratio=0.5, ck_ratio=0.5): 
    
    # Kernel 
    pk = cr.kernels.PseudotimeKernel(adata, time_key='palantir_pseudotime')
    ck = cr.kernels.ConnectivityKernel(adata)
    
    # Transission matrix
    pk = pk.compute_transition_matrix()
    ck = ck.compute_transition_matrix()
    
    k = pk_ratio*pk + ck_ratio*ck
    
    return k 

In [ ]:
def estimator_workflow(k, n_components=10): 
    
    from cellrank.estimators import GPCCA
    g = GPCCA(k)
    
    g.compute_schur(n_components=n_components)
    
    return(g)

# CD4+ T-cells

## SCVI integration 

In [ ]:
if(False): 
    
    # Leiden subset annotation filter 
    cell_type_filter_1 = ['$T_{h, cm/naive}$', '$T_{h, em/effector}$', '$T_{h, memory/effector}$']
    
    # Leiden subset cycling cells CellTypist annotation filter
    cell_type_filter_2 = set([i for i in adata.obs['celltypist_immun_low'] if 'helper T cells' in i])
    
    # Filter cells 
    adata_0 = adata[(adata.obs['cell_type_leiden_subset'].isin(cell_type_filter_1) | ((adata.obs['cell_type_leiden_subset']=='$Cycling$') & (adata.obs['celltypist_immun_low'].isin(cell_type_filter_2))))].copy()
    
    # SCVI integration 
    adata_0 = scvi_integrate(adata_0, model_dir='data/object/pseudotime/model_0', file='data/object/pseudotime/scvi_0.h5ad')
    
else: 
        
    adata_0 = sc.read_h5ad('data/object/pseudotime/scvi_0.h5ad')

## PAGA embedding and pseudotime 

In [ ]:
if(False): 
    
    # Set raw counts 
    adata_0 = adata_0.raw.to_adata()
    adata_0.raw = adata_0
    
    # PAGA graph and UMAP 
    adata_0 = paga_dim(adata_0, n_neighbors=15, resolution=2)

    # Palantir imputation and pseudotime 
    adata_0 = palantir_workflow(adata_0,'$T_{h, cm/naive}$', 'cell_type_leiden_subset', n_components=15, knn=30)

    # Save results 
    adata_0.write_h5ad('data/object/pseudotime/scvi_pt_0.h5ad')
    
else: 
    
    adata_0 = sc.read_h5ad('data/object/pseudotime/scvi_pt_0.h5ad')

## Module score 

In [ ]:
if(False): 
    
    # Module score 
    adata_0 = module_score(adata=adata_0)
    
    # Save results 
    adata_0.write_h5ad('data/object/pseudotime/scvi_pt_0.h5ad')
    
else: 
    
    adata_0 = sc.read_h5ad('data/object/pseudotime/scvi_pt_0.h5ad')

## Plot results 

In [ ]:
# Plot results 
pl_summary(adata_0.copy(), layer='data_magic')

## CellRank lineage analysis 

In [ ]:
if(False): 
    
    # Kernel 
    k_0 = kernel_workflow(adata_0, pk_ratio=0.9, ck_ratio=0.1)
    k_0.write('data/object/pseudotime/k_0.pkl')
    
    # Estimator and macrostates
    g_0 = estimator_workflow(k_0)
    g_0.compute_macrostates(n_states=10, cluster_key='leiden')
    
    # Save estimator 
    g_0.write('data/object/pseudotime/g_0.pkl', write_adata=True)
    
else: 
    
    with open('data/object/pseudotime/k_0.pkl', 'rb') as file:
        k_0 = pickle.load(file)
        
    with open('data/object/pseudotime/g_0.pkl', 'rb') as file:
        g_0 = pickle.load(file)

In [ ]:
k_0.plot_projection(basis='X_umap', color=['cell_type_leiden_subset', 'hto_demux_class', 'time_point', 'genotype_class', 'has_ir', 'palantir_pseudotime'], legend_loc='right', recompute=True, density=1.5, linewidth=3, arrow_size=1.5)

In [ ]:
g_0.plot_macrostates(which='all', same_plot=True, discrete=True, figsize=(5, 5))
g_0.plot_macrostates(which='all', same_plot=False, discrete=False, ncols=8, figsize=(5, 5))

In [ ]:
g_0.set_terminal_states(states=['0', '2, 13'])
g_0.set_initial_states(['1, 6'])

In [ ]:
g_0.compute_fate_probabilities()
g_0.plot_fate_probabilities(same_plot=False, legend_loc='right', figsize=(5, 5))

In [ ]:
model_0 = cr.models.GAMR(g_0.adata, n_knots=6, smoothing_penalty=10.0)

In [ ]:
cr.pl.gene_trends(
    
    g_0.adata,
    model=model_0,
    data_key='data_magic',
    genes=['CREM', 'LMNA', 'EZR', 'ZFP36', 'CD69'],
    same_plot=True,
    ncols=5,
    time_key='palantir_pseudotime',
    hide_cells=True,
    weight_threshold=(1e-3, 1e-3)
    
)

# CD8+ T-cells

## SCVI integration 

In [ ]:
if(False): 
    
    # Leiden subset annotation filter 
    cell_type_filter_1 = ['$T_{c, cm/naive}$', '$T_{c, em/effector}$', '$T_{c, memory/effector}$']
    
    # Leiden subset cycling cells CellTypist annotation filter
    cell_type_filter_2 = set([i for i in adata.obs['celltypist_immun_low'] if 'cytotoxic T cells' in i])
    
    # Filter cells 
    adata_1 = adata[(adata.obs['cell_type_leiden_subset'].isin(cell_type_filter_1) | ((adata.obs['cell_type_leiden_subset']=='$Cycling$') & (adata.obs['celltypist_immun_low'].isin(cell_type_filter_2)))) & (adata.obs['celltypist_immun_high']=='T cells')].copy()

    # SCVI integration 
    adata_1 = scvi_integrate(adata_1, model_dir='data/object/pseudotime/model_1', file='data/object/pseudotime/scvi_1.h5ad')

else: 
    
    adata_1 = sc.read_h5ad('data/object/pseudotime/scvi_1.h5ad')

## PAGA embedding and pseudotime 

In [ ]:
if(False): 
    
    # Set raw counts 
    adata_1 = adata_1.raw.to_adata()
    adata_1.raw = adata_1

    # PAGA graph and UMAP 
    adata_1 = paga_dim(adata_1, n_neighbors=15, resolution=2)

    # Palantir imputation and pseudotime 
    adata_1 = palantir_workflow(adata_1, celltype='$T_{c, cm/naive}$', celltype_column='cell_type_leiden_subset', n_components=15, knn=30)
    
    # Save results 
    adata_1.write_h5ad('data/object/pseudotime/scvi_pt_1.h5ad')
    
    # Plot results 
    pl_summary(adata_1.copy())
    
else: 
    
    adata_1 = sc.read_h5ad('data/object/pseudotime/scvi_pt_1.h5ad')

## Module score 

In [ ]:
if(True): 
    
    # Module score 
    adata_1 = module_score(adata=adata_1)
    
    # Save results 
    adata_1.write_h5ad('data/object/pseudotime/scvi_pt_1.h5ad')
    
else: 
    
    adata_1 = sc.read_h5ad('data/object/pseudotime/scvi_pt_1.h5ad')

## Plot results 

In [ ]:
# Plot results 
pl_summary(adata_1.copy(), layer='data_magic')

## CellRank lineage analysis 

In [ ]:
if(False): 
    
    # Kernel 
    k_1 = kernel_workflow(adata_1, pk_ratio=0.9, ck_ratio=0.1)
    k_1.write('data/object/pseudotime/k_1.pkl')
    
    # Estimator and macrostates
    g_1 = estimator_workflow(k_1)
    g_1.compute_macrostates(n_states=10, cluster_key='leiden')
    
    # Save estimator 
    g_1.write('data/object/pseudotime/g_1.pkl', write_adata=True)
    
else: 
    
    with open('data/object/pseudotime/k_1.pkl', 'rb') as file:
        k_1 = pickle.load(file)
        
    with open('data/object/pseudotime/g_1.pkl', 'rb') as file:
        g_1 = pickle.load(file)

In [ ]:
k_1.plot_projection(basis='X_umap', color=['cell_type_leiden_subset', 'hto_demux_class', 'time_point', 'genotype_class', 'has_ir', 'palantir_pseudotime'], legend_loc='right', recompute=True, density=1.5, linewidth=3, arrow_size=1.5)

In [ ]:
g_1.plot_macrostates(which='all', same_plot=True, discrete=True, figsize=(5, 5))
g_1.plot_macrostates(which='all', same_plot=False, discrete=False, ncols=8, figsize=(5, 5))

In [ ]:
g_1.set_terminal_states(states=['2', '4', '8', '15', '25', '12', '10'])
g_1.set_initial_states(['0'])

In [ ]:
g_1.compute_fate_probabilities()
g_1.plot_fate_probabilities(same_plot=False, legend_loc='right', figsize=(5, 5))

In [ ]:
model_1 = cr.models.GAMR(g_1.adata, n_knots=6, smoothing_penalty=10.0)

In [ ]:
cr.pl.gene_trends(
    
    g_1.adata,
    model=model_1,
    data_key='data_magic',
    genes=['CREM', 'LMNA', 'EZR', 'ZFP36', 'CD69', 'MKI67', 'PCNA'],
    same_plot=True,
    ncols=5,
    time_key='palantir_pseudotime',
    hide_cells=True,
    weight_threshold=(1e-3, 1e-3)
    
)

In [ ]:
cr.pl.gene_trends(
    
    g_1.adata,
    model=model_1,
    data_key='data_magic',
    genes=['CCR6', 'IL7R', 'IL17A', 'MAF', 'RUNX3', 'ZNF683'],
    same_plot=True,
    ncols=5,
    time_key='palantir_pseudotime',
    hide_cells=True,
    weight_threshold=(1e-3, 1e-3)
    
)